In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score, roc_auc_score, roc_curve, auc
from sklearn.model_selection import learning_curve, LeaveOneOut, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE, RFECV, SelectFromModel, SelectFpr, VarianceThreshold, chi2
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier

In [2]:
dataset = "C:/Users/tamer/Documents/PhD/ML/enhanced_metabolome.xlsx"
df = pd.read_excel(dataset, sheet_name = 'Paul_50%')
df_val = pd.read_excel(dataset, sheet_name = 'Saqib_50%')

#dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_paul.xlsx"
#dataset_val_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_saqib.xlsx"

#df = pd.read_excel(dataset_total_path)
#df_val = pd.read_excel(dataset_val_path)

print(df.shape)
print(df_val.shape)

(24, 648)
(12, 648)


In [3]:
Class_A = 'SP'
Class_B = 'LP'

df = df[(df["Class"] == Class_A) | (df["Class"] == Class_B)]
df_val = df_val[(df_val["Class"] == Class_A) | (df_val["Class"] == Class_B)]

In [4]:
def encodage(df):
    code = {
    'LP' : 1,
    'SP' : 10,
    'LN' : 0,
    'SN' : 4
}
# Appliquer ce dictionnaire aux colonnes du dataset, avec la fonction map
    for col in df.select_dtypes('object'):
        df[col] = df[col].map(code)

    return df


def preprocessing(df):
    df = encodage(df)

    X = df.drop(['Class'], axis = 1)
    y = df['Class']

    # compter le nombre d'échantillons restants dans le dataset après avoir été inputé
    print(y.value_counts())

    return X, y

In [16]:
X_total, y_total = preprocessing(df)
X_val, y_val = preprocessing(df_val)

Class
1     6
10    6
Name: count, dtype: int64
Class
1     3
10    3
Name: count, dtype: int64


In [17]:
print("var before log2 transormation : " , X_total.var(axis=0).mean())
log2_transformer = FunctionTransformer(lambda x: np.log2(x + 1))
X_total = log2_transformer.fit_transform(X_total)
X_val = log2_transformer.fit_transform(X_val)
print("var after log2 transormation : " , X_total.var(axis=0).mean())

var before log2 transormation :  2.991038206514167e+17
var after log2 transormation :  5.107260649767679


In [18]:
def evaluation(model):
    model.fit(X_total, y_total)
    ypred = model.predict(X_val)

    print(confusion_matrix(y_val, ypred))
    print(classification_report(y_val, ypred))

## Définition modèle

In [22]:
vars_ = X_total.var(axis=0)
q = np.quantile(vars_, 0.50) 
selector = VarianceThreshold(threshold = q)

preprocessor = make_pipeline(selector)

LR_L1 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l1', C=10, random_state = 0, solver = 'liblinear'))
evaluation(LR_L1)

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         3
          10       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



In [9]:
#hyper_params = {'logisticregression__C': [0.1, 1, 10], 'selectkbest__k' : range(100, 350)}
#grid = GridSearchCV(LR_L1, cv = LeaveOneOut(), param_grid=hyper_params, scoring='accuracy')
#grid.fit(X_total, y_total)
#print(grid.best_params_)
#evaluation(grid.best_estimator_)

## ROC/AUC curve

In [20]:
# Fit model
LR_L1.fit(X_total, y_total)

# Get predicted probabilities for the positive class
y_score = LR_L1.predict_proba(X_total)[:, 1]

# ROC
fpr, tpr, thresholds = roc_curve(y_total, y_score)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve (in-sample)")
plt.legend(loc="lower right")

plt.savefig("C:/Users/tamer/Documents/PhD/ML/Performance/ROC_curve.pdf", format="pdf", bbox_inches="tight")   
plt.savefig("C:/Users/tamer/Documents/PhD/ML/Performance/ROC_curve.png", dpi=600, bbox_inches="tight")   


plt.show()

ValueError: y_true takes value in {1, 10} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.

In [10]:
LR_L1 = make_pipeline(StandardScaler(), LogisticRegression(penalty='l1', random_state = 0, solver = 'liblinear'))
evaluation(LR_L1)

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         3
          10       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



In [11]:
model = LR_L1  # your new pipeline
y_arr = np.asarray(y_total)

n_boot = 1000
n_features = X_total.shape[1]
coef_mat = np.zeros((n_boot, n_features))

for b in range(n_boot):
    # Stratified bootstrap sampling
    idx_R = np.where(y_arr == 1)[0]
    idx_S = np.where(y_arr == 10)[0]

    boot_R = np.random.choice(idx_R, size=len(idx_R), replace=True)
    boot_S = np.random.choice(idx_S, size=len(idx_S), replace=True)

    boot_idx = np.concatenate([boot_R, boot_S])
    Xb = X_total.iloc[boot_idx]
    yb = y_arr[boot_idx]

    # Fit model
    model.fit(Xb, yb)

    # Extract L1 coefficients directly
    coefs = model.named_steps["logisticregression"].coef_[0]

    # Save to matrix
    coef_mat[b, :] = coefs

In [12]:
print(coef_mat.shape)
print("Any non-zero coefficients?", np.any(coef_mat != 0))
print("Mean #nonzero per bootstrap:", np.mean(np.sum(coef_mat != 0, axis=1)))
print("Std  #nonzero per bootstrap:", np.std(np.sum(coef_mat != 0, axis=1)))

(1000, 647)
Any non-zero coefficients? True
Mean #nonzero per bootstrap: 71.604
Std  #nonzero per bootstrap: 19.93005730046956


In [13]:
coef_df = pd.DataFrame(coef_mat, columns=X_total.columns)

summary = pd.DataFrame({
    "metabolite": X_total.columns,
    "mean_coef": coef_df.mean(axis=0),
    "std_coef": coef_df.std(axis=0),
    # fraction of bootstraps where coefficient != 0 (usually equals "passed variance filter")
    "nonzero_freq": (coef_df != 0).mean(axis=0),
    # fraction of bootstraps where coefficient is positive / negative (ignoring zeros)
    "pos_freq": (coef_df > 0).mean(axis=0),
    "neg_freq": (coef_df < 0).mean(axis=0),
})

# sign consistency (close to 1 means always same sign; close to 0 means flips)
summary["sign_consistency"] = (summary["pos_freq"] - summary["neg_freq"]).abs()

# optional: "signal-to-noise" style ranking
summary["abs_mean_over_sd"] = summary["mean_coef"].abs() / (summary["std_coef"] + 1e-12)

# sort: first by stability, then by magnitude/stability ratio
summary = summary.sort_values(
    ["nonzero_freq", "sign_consistency", "abs_mean_over_sd"],
    ascending=[False, False, False]
).reset_index(drop=True)

print(summary.head(20))

                                           metabolite  mean_coef  std_coef  \
0                                      Adiantifoline;  -0.044759  0.039588   
1                                         Myricoside;   0.044342  0.035322   
2   1-(acetyloxy)-3-hydroxy-6,8a-dimethyl-7-oxo-3-...   0.046673  0.037605   
3                                       Thalicarpine;  -0.045947  0.036835   
4   4-[3-(4-hydroxy-3-methoxybenzoyl)-2,3-dimethyl...   0.041297  0.067862   
5   5-hydroxy-3-(4-methoxyphenyl)-7-[(3,4,5-trihyd...   0.034604  0.031688   
6   4-{[(morpholinosulfonyl)methyl]sulfonyl}morpho...   0.030823  0.029459   
7                                             Didymin   0.041089  0.036987   
8    [4-(2-methoxyphenyl)piperidino](phenyl)methanone   0.037968  0.033462   
9                                 trans-Aconitic acid  -0.035511  0.032662   
10  N-[3,5-bis(trifluoromethyl)phenyl]-N'-[2-(2-ox...   0.029086  0.029874   
11  N'-methyl-N'-(3-nitro-2-pyridyl)-(methylsulfon...   0.030931

In [14]:
summary.to_excel("C:/Users/tamer/Documents/PhD/ML/ML_coefs/bootstrap.xlsx", index=False)
print("Saved: bootstrap_LR_L1_coeff_stability.xlsx")

Saved: bootstrap_LR_L1_coeff_stability.xlsx


## Bootstrap

### With preprocessor

In [23]:
model = LR_L1

y_arr = np.asarray(y_total)  # <- important fix

n_boot = 1000
n_features = X_total.shape[1]
coef_mat = np.zeros((n_boot, n_features))

for b in range(n_boot):

    idx_R = np.where(y_arr == 10)[0]
    idx_S = np.where(y_arr == 1)[0]

    boot_R = np.random.choice(idx_R, size=len(idx_R), replace=True)
    boot_S = np.random.choice(idx_S, size=len(idx_S), replace=True)

    boot_idx = np.concatenate([boot_R, boot_S])

    Xb = X_total.iloc[boot_idx]
    yb = y_arr[boot_idx]  # <- now positional, no KeyError

    model.fit(Xb, yb)

    coefs = model.named_steps["logisticregression"].coef_[0]
    #support = model.named_steps['pipeline'].named_steps['selectkbest'].get_support()
    support = model.named_steps['pipeline'].named_steps['variancethreshold'].get_support()

    full_coef = np.zeros(n_features)
    full_coef[support] = coefs

    coef_mat[b, :] = full_coef

In [24]:
print(coef_mat.shape)
print("Any non-zero coefficients?", np.any(coef_mat != 0))
print("Mean #nonzero per bootstrap:", np.mean(np.sum(coef_mat != 0, axis=1)))
print("Std  #nonzero per bootstrap:", np.std(np.sum(coef_mat != 0, axis=1)))

(1000, 647)
Any non-zero coefficients? True
Mean #nonzero per bootstrap: 57.834
Std  #nonzero per bootstrap: 11.011014667141263


In [ ]:
coef_df = pd.DataFrame(coef_mat, columns=X_total.columns)

summary = pd.DataFrame({
    "metabolite": X_total.columns,
    "mean_coef": coef_df.mean(axis=0),
    "std_coef": coef_df.std(axis=0),
    # fraction of bootstraps where coefficient != 0 (usually equals "passed variance filter")
    "nonzero_freq": (coef_df != 0).mean(axis=0),
    # fraction of bootstraps where coefficient is positive / negative (ignoring zeros)
    "pos_freq": (coef_df > 0).mean(axis=0),
    "neg_freq": (coef_df < 0).mean(axis=0),
})

# sign consistency (close to 1 means always same sign; close to 0 means flips)
summary["sign_consistency"] = (summary["pos_freq"] - summary["neg_freq"]).abs()

# optional: "signal-to-noise" style ranking
summary["abs_mean_over_sd"] = summary["mean_coef"].abs() / (summary["std_coef"] + 1e-12)

# sort: first by stability, then by magnitude/stability ratio
summary = summary.sort_values(
    ["nonzero_freq", "sign_consistency", "abs_mean_over_sd"],
    ascending=[False, False, False]
).reset_index(drop=True)

print(summary.head(20))

In [ ]:
summary.to_excel("C:/Users/tamer/Documents/PhD/ML/ML_coefs/bootstrap.xlsx", index=False)
print("Saved: bootstrap_LR_L1_coeff_stability.xlsx")